# 96. nomic-embed-text-v2-moe 総合評価

## 目的
- nomic-ai/nomic-embed-text-v2-moe (MoEアーキテクチャ, 768D) を多角的に評価
- NB91〜95の観点を1つのノートブックに凝縮

## 評価項目
| Sec | 内容 | 参考NB |
|-----|------|--------|
| 0 | Setup | NB91 |
| 1 | データ準備 (Wikipedia JA/EN 10K) | NB91 |
| 2 | STSデータセット読み込み | NB92 |
| 3 | ヘルパー関数定義 | NB91/92/95 |
| 4 | GPU Embedding生成と速度計測 | NB91 |
| 5 | Embedding品質分析 | NB91 |
| 6 | STSベンチマーク評価 | NB92 |
| 7 | CPU推論速度比較 | NB95 |
| 8 | ITQ-LSH適合性 (ビット長最適化) | NB94 |
| 9 | Embedding・ハッシュ保存 | NB91/94 |
| 10 | 総合サマリー | NB91 |

## モデル特性
| 項目 | 値 |
|------|----|
| モデル | nomic-ai/nomic-embed-text-v2-moe |
| アーキテクチャ | Mixture of Experts |
| 次元 | 768 (Matryoshka対応) |
| プレフィックス | search_document: / search_query: |

## 0. Setup

In [1]:
import numpy as np
import time
import gc
import torch
import sys
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from scipy.stats import spearmanr, pearsonr

sys.path.insert(0, '../src')
from itq_lsh import ITQLSH

DATA_DIR = Path('../data')
np.random.seed(42)

MODEL_ID = 'nomic-ai/nomic-embed-text-v2-moe'
MODEL_KEY = 'nomic_v2_moe'
N_SAMPLES = 10000
MAX_CHARS = 500

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 4090
VRAM: 23.5 GB


## 1. データ準備 (Wikipedia JA/EN 10K)

In [2]:
from datasets import load_dataset

def collect_wikipedia(lang_code: str, n_samples: int = N_SAMPLES) -> list[str]:
    """Wikipedia からテキストを収集（プレフィックスなし）"""
    wiki = load_dataset(
        'wikimedia/wikipedia',
        f'20231101.{lang_code}',
        split='train',
        streaming=True
    )
    documents = []
    for i, item in enumerate(tqdm(wiki, total=n_samples, desc=f'Wikipedia {lang_code}')):
        if i >= n_samples:
            break
        text = item['text'][:MAX_CHARS].strip()
        if len(text) < 50:
            continue
        documents.append(text)
    print(f'Collected {len(documents):,} documents ({lang_code})')
    return documents

print('=== Japanese ===')
docs_ja = collect_wikipedia('ja')
print(f'\n=== English ===')
docs_en = collect_wikipedia('en')

print(f'\nJA: {len(docs_ja):,} docs, sample: {docs_ja[0][:80]}...')
print(f'EN: {len(docs_en):,} docs, sample: {docs_en[0][:80]}...')

=== Japanese ===


Wikipedia ja:   0%|          | 0/10000 [00:00<?, ?it/s]

Wikipedia ja:   0%|          | 1/10000 [00:04<11:17:17,  4.06s/it]

Wikipedia ja:  10%|▉         | 999/10000 [00:04<00:26, 339.11it/s]

Wikipedia ja:  16%|█▌        | 1586/10000 [00:07<00:33, 254.15it/s]

...

Wikipedia en:  77%|███████▋  | 7679/10000 [00:07<00:00, 3415.19it/s]

Wikipedia en:  88%|████████▊ | 8826/10000 [00:07<00:00, 4423.48it/s]

Wikipedia en:  99%|█████████▉| 9948/10000 [00:07<00:00, 5454.39it/s]

Wikipedia en: 100%|██████████| 10000/10000 [00:07<00:00, 1265.86it/s]

Collected 10,000 documents (en)

JA: 9,990 docs, sample: アンパサンド（&, ）は、並立助詞「…と…」を意味する記号である。ラテン語で「…と…」を表す接続詞 "et" の合字を起源とする。現代のフォントでも、Trebu...
EN: 10,000 docs, sample: Anarchism is a political philosophy and movement that is skeptical of all justif...


## 2. STSデータセット読み込み

In [3]:
import json
import urllib.request
import io

# JSTS (Japanese STS)
jsts_url = 'https://raw.githubusercontent.com/yahoojapan/JGLUE/v1.1.0/datasets/jsts-v1.1/valid-v1.1.json'
with urllib.request.urlopen(jsts_url) as resp:
    jsts_raw = [json.loads(line) for line in resp.read().decode('utf-8').strip().split('\n')]
print(f'JSTS: {len(jsts_raw)} pairs')

# JSICK (Japanese SICK)
jsick_url = 'https://raw.githubusercontent.com/verypluming/JSICK/b3034994192fae2f41b5937bcf69544e4282fc39/jsick/jsick.tsv'
with urllib.request.urlopen(jsick_url) as resp:
    jsick_df = pd.read_csv(io.StringIO(resp.read().decode('utf-8')), delimiter='\t')
jsick_test = jsick_df[jsick_df['data'] == 'test'].reset_index(drop=True)
print(f'JSICK: {len(jsick_test)} pairs')

# STS-B (English)
stsb_ds = load_dataset('sentence-transformers/stsb', split='test')
print(f'STS-B: {len(stsb_ds)} pairs')

sts_datasets = {
    'JSTS': {
        'sentences1': [d['sentence1'] for d in jsts_raw],
        'sentences2': [d['sentence2'] for d in jsts_raw],
        'scores': [d['label'] for d in jsts_raw],
    },
    'JSICK': {
        'sentences1': jsick_test['sentence_A_Ja'].tolist(),
        'sentences2': jsick_test['sentence_B_Ja'].tolist(),
        'scores': jsick_test['relatedness_score_Ja'].tolist(),
    },
    'STS-B': {
        'sentences1': stsb_ds['sentence1'],
        'sentences2': stsb_ds['sentence2'],
        'scores': stsb_ds['score'],
    },
}

for name, data in sts_datasets.items():
    print(f'{name}: {len(data["sentences1"])} pairs, score range: [{min(data["scores"]):.2f}, {max(data["scores"]):.2f}]')

JSTS: 1457 pairs


JSICK: 4927 pairs


STS-B: 1379 pairs
JSTS: 1457 pairs, score range: [0.00, 5.00]
JSICK: 4927 pairs, score range: [1.00, 5.00]
STS-B: 1379 pairs, score range: [0.00, 1.00]


## 3. ヘルパー関数定義

In [4]:
def clear_gpu():
    """GPU メモリを解放"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def evaluate_embeddings(embeddings: np.ndarray, name: str, n_pairs: int = 10000) -> dict:
    """Embedding品質の評価指標を計算"""
    n = len(embeddings)
    rng = np.random.default_rng(42)
    idx1 = rng.choice(n, n_pairs, replace=True)
    idx2 = rng.choice(n, n_pairs, replace=True)
    mask = idx1 != idx2
    idx1, idx2 = idx1[mask], idx2[mask]

    cos_sims = np.sum(embeddings[idx1] * embeddings[idx2], axis=1)

    centered = embeddings - embeddings.mean(axis=0)
    cov = centered.T @ centered / n
    eigenvalues = np.linalg.eigvalsh(cov)[::-1]
    top10_ratio = eigenvalues[:10].sum() / eigenvalues.sum()
    condition_number = eigenvalues[0] / eigenvalues[-1] if eigenvalues[-1] > 1e-10 else float('inf')

    return {
        'model': name,
        'dim': embeddings.shape[1],
        'cos_mean': float(np.mean(cos_sims)),
        'cos_std': float(np.std(cos_sims)),
        'cos_min': float(np.min(cos_sims)),
        'cos_max': float(np.max(cos_sims)),
        'top10_var_ratio': float(top10_ratio),
        'condition_number': float(condition_number),
    }


def compute_sts_metrics(embeddings1: np.ndarray, embeddings2: np.ndarray,
                        gold_scores: list[float]) -> dict:
    """コサイン類似度とゴールドスコアのSpearman/Pearson相関を計算"""
    cos_sims = np.sum(embeddings1 * embeddings2, axis=1)
    gold = np.array(gold_scores)
    sp, sp_pval = spearmanr(cos_sims, gold)
    pe, pe_pval = pearsonr(cos_sims, gold)
    return {'spearman': float(sp), 'pearson': float(pe), 'sp_pval': float(sp_pval), 'pe_pval': float(pe_pval)}


def benchmark_encode(encode_fn, docs: list[str], n_runs: int = 3, warmup: int = 1) -> dict:
    """エンコード関数のベンチマーク"""
    for _ in range(warmup):
        _ = encode_fn(docs[:10])
    times = []
    embeddings = None
    for _ in range(n_runs):
        start = time.time()
        emb = encode_fn(docs)
        elapsed = time.time() - start
        times.append(elapsed)
        if embeddings is None:
            embeddings = emb
    avg_time = np.mean(times)
    return {
        'time': avg_time,
        'std': np.std(times),
        'docs_per_sec': len(docs) / avg_time,
        'ms_per_doc': avg_time / len(docs) * 1000,
        'embeddings': embeddings,
    }


def quick_itq_eval(embeddings: np.ndarray, model_name: str, n_bits: int = 128,
                   n_pairs: int = 10000) -> dict:
    """ITQ-LSHのハッシュ品質を簡易評価"""
    itq = ITQLSH(n_bits=n_bits, n_iterations=50, seed=42)
    itq.fit(embeddings)
    hashes = itq.transform(embeddings)

    rng = np.random.default_rng(42)
    n = len(embeddings)
    idx1 = rng.choice(n, n_pairs, replace=True)
    idx2 = rng.choice(n, n_pairs, replace=True)
    mask = idx1 != idx2
    idx1, idx2 = idx1[mask], idx2[mask]

    cos_sims = np.sum(embeddings[idx1] * embeddings[idx2], axis=1)
    ham_dists = np.sum(hashes[idx1] != hashes[idx2], axis=1)
    corr, pval = spearmanr(ham_dists, cos_sims)

    return {
        'model': model_name,
        'n_bits': n_bits,
        'spearman': corr,
        'pvalue': pval,
        'ham_mean': float(np.mean(ham_dists)),
        'ham_std': float(np.std(ham_dists)),
        'itq': itq,
        'hashes': hashes,
    }

print('Helper functions ready.')

Helper functions ready.


## 4. GPU Embedding生成と速度計測

In [5]:
from sentence_transformers import SentenceTransformer

print(f'Loading {MODEL_ID}...')
model = SentenceTransformer(MODEL_ID, device='cuda', trust_remote_code=True)
print(f'Embedding dim: {model.get_sentence_embedding_dimension()}')

# JA (search_document: プレフィックス)
docs_ja_prefixed = [f'search_document: {d}' for d in docs_ja]
start = time.time()
emb_ja = model.encode(docs_ja_prefixed, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
time_ja = time.time() - start
print(f'JA: {emb_ja.shape}, {len(docs_ja)/time_ja:.0f} docs/sec')

# EN
docs_en_prefixed = [f'search_document: {d}' for d in docs_en]
start = time.time()
emb_en = model.encode(docs_en_prefixed, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
time_en = time.time() - start
print(f'EN: {emb_en.shape}, {len(docs_en)/time_en:.0f} docs/sec')

speed_gpu = {
    'ja_docs_sec': len(docs_ja) / time_ja,
    'en_docs_sec': len(docs_en) / time_en,
}
print(f'\nGPU速度: JA {speed_gpu["ja_docs_sec"]:.0f} docs/sec, EN {speed_gpu["en_docs_sec"]:.0f} docs/sec')

Loading nomic-ai/nomic-embed-text-v2-moe...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/554 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Embedding dim: 768


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

JA: (9990, 768), 217 docs/sec


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

EN: (10000, 768), 517 docs/sec

GPU速度: JA 217 docs/sec, EN 517 docs/sec


## 5. Embedding品質分析

In [6]:
result_ja = evaluate_embeddings(emb_ja, MODEL_KEY)
result_en = evaluate_embeddings(emb_en, MODEL_KEY)

print('='*70)
print(f'Embedding品質: {MODEL_KEY}')
print('='*70)
for lang, r in [('JA', result_ja), ('EN', result_en)]:
    print(f'\n--- {lang} ---')
    print(f'  Dim: {r["dim"]}')
    print(f'  Cosine mean: {r["cos_mean"]:.4f} (低いほど等方的)')
    print(f'  Cosine std:  {r["cos_std"]:.4f}')
    print(f'  Cosine range: [{r["cos_min"]:.4f}, {r["cos_max"]:.4f}]')
    print(f'  Top10 var ratio: {r["top10_var_ratio"]:.4f}')
    print(f'  Condition number: {r["condition_number"]:.1f}')

# NB91参考値
print('\n--- NB91参考値 (cos_mean) ---')
ref = {
    'e5_base': (0.706, 0.594),
    'gemma_300m': (0.360, 0.206),
    'qwen3_06b': (0.494, 0.384),
    'bge_m3': (0.605, 0.459),
    'ruri_v3': (0.625, 0.384),
}
print(f'{"model":<15} {"JA":>8} {"EN":>8}')
for m, (ja, en) in ref.items():
    print(f'{m:<15} {ja:>8.4f} {en:>8.4f}')
print(f'{MODEL_KEY:<15} {result_ja["cos_mean"]:>8.4f} {result_en["cos_mean"]:>8.4f}')

Embedding品質: nomic_v2_moe

--- JA ---
  Dim: 768
  Cosine mean: 0.2731 (低いほど等方的)
  Cosine std:  0.0924
  Cosine range: [0.0604, 0.8555]
  Top10 var ratio: 0.2596
  Condition number: 385960544.0

--- EN ---
  Dim: 768
  Cosine mean: 0.1715 (低いほど等方的)
  Cosine std:  0.0741
  Cosine range: [-0.0544, 0.6396]
  Top10 var ratio: 0.1731
  Condition number: 193730608.0

--- NB91参考値 (cos_mean) ---
model                 JA       EN
e5_base           0.7060   0.5940
gemma_300m        0.3600   0.2060
qwen3_06b         0.4940   0.3840
bge_m3            0.6050   0.4590
ruri_v3           0.6250   0.3840
nomic_v2_moe      0.2731   0.1715


## 6. STSベンチマーク評価

In [7]:
# nomicモデルはまだGPU上にある
def encode_nomic(texts):
    return model.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=False)

sts_results = []
print(f'STS Benchmark: {MODEL_KEY}')
print('='*60)

for ds_name, ds_data in sts_datasets.items():
    s1 = [f'search_query: {s}' for s in ds_data['sentences1']]
    s2 = [f'search_query: {s}' for s in ds_data['sentences2']]
    scores = ds_data['scores']

    start = time.time()
    emb1 = encode_nomic(s1)
    emb2 = encode_nomic(s2)
    elapsed = time.time() - start

    metrics = compute_sts_metrics(emb1, emb2, scores)
    metrics['dataset'] = ds_name
    metrics['time_sec'] = elapsed
    sts_results.append(metrics)
    print(f'  {ds_name}: Spearman={metrics["spearman"]:.4f}, Pearson={metrics["pearson"]:.4f} ({elapsed:.1f}s)')

# GPUモデルを解放
del model
clear_gpu()
print('\nGPU model released.')

# NB92参考値との比較
print('\n--- NB92参考値 (Spearman) ---')
ref_sts = {
    'e5_base':    {'JSTS': 0.8326, 'JSICK': 0.7681, 'STS-B': 0.8488},
    'gemma_300m': {'JSTS': 0.7779, 'JSICK': 0.7629, 'STS-B': 0.8463},
    'qwen3_06b':  {'JSTS': 0.8452, 'JSICK': 0.7925, 'STS-B': 0.8705},
    'bge_m3':     {'JSTS': 0.8338, 'JSICK': 0.7703, 'STS-B': 0.8427},
    'ruri_v3':    {'JSTS': 0.8432, 'JSICK': 0.7776, 'STS-B': 0.7771},
}
print(f'{"model":<15} {"JSTS":>8} {"JSICK":>8} {"STS-B":>8}')
for m, vals in ref_sts.items():
    print(f'{m:<15} {vals["JSTS"]:>8.4f} {vals["JSICK"]:>8.4f} {vals["STS-B"]:>8.4f}')
nomic_row = {r['dataset']: r['spearman'] for r in sts_results}
print(f'{MODEL_KEY:<15} {nomic_row.get("JSTS", 0):>8.4f} {nomic_row.get("JSICK", 0):>8.4f} {nomic_row.get("STS-B", 0):>8.4f}')

STS Benchmark: nomic_v2_moe


  JSTS: Spearman=0.7726, Pearson=0.8173 (1.6s)


  JSICK: Spearman=0.8166, Pearson=0.8245 (5.0s)


  STS-B: Spearman=0.8343, Pearson=0.8414 (1.5s)

GPU model released.

--- NB92参考値 (Spearman) ---
model               JSTS    JSICK    STS-B
e5_base           0.8326   0.7681   0.8488
gemma_300m        0.7779   0.7629   0.8463
qwen3_06b         0.8452   0.7925   0.8705
bge_m3            0.8338   0.7703   0.8427
ruri_v3           0.8432   0.7776   0.7771
nomic_v2_moe      0.7726   0.8166   0.8343


## 7. CPU推論速度比較

In [8]:
# CPU推論（1000件に絞ってベンチマーク）
N_CPU_BENCH = 1000
docs_ja_cpu = docs_ja[:N_CPU_BENCH]
docs_en_cpu = docs_en[:N_CPU_BENCH]

print(f'Loading {MODEL_ID} on CPU...')
model_cpu = SentenceTransformer(MODEL_ID, device='cpu', trust_remote_code=True)

def encode_nomic_cpu(docs):
    prefixed = [f'search_document: {d}' for d in docs]
    return model_cpu.encode(prefixed, batch_size=32, normalize_embeddings=True, show_progress_bar=False)

print('\n--- JA ---')
res_cpu_ja = benchmark_encode(encode_nomic_cpu, docs_ja_cpu)
print(f'  {res_cpu_ja["docs_per_sec"]:.1f} docs/sec, {res_cpu_ja["ms_per_doc"]:.1f} ms/doc')

print('--- EN ---')
res_cpu_en = benchmark_encode(encode_nomic_cpu, docs_en_cpu)
print(f'  {res_cpu_en["docs_per_sec"]:.1f} docs/sec, {res_cpu_en["ms_per_doc"]:.1f} ms/doc')

del model_cpu
clear_gpu()

# GPU vs CPU 比較
print('\n' + '='*60)
print('GPU vs CPU 速度比較')
print('='*60)
print(f'{"":<6} {"GPU (docs/s)":>14} {"CPU (docs/s)":>14} {"GPU/CPU ratio":>14}')
print('-' * 50)
for lang, gpu_speed, cpu_res in [
    ('JA', speed_gpu['ja_docs_sec'], res_cpu_ja),
    ('EN', speed_gpu['en_docs_sec'], res_cpu_en),
]:
    ratio = gpu_speed / cpu_res['docs_per_sec']
    print(f'{lang:<6} {gpu_speed:>13.0f} {cpu_res["docs_per_sec"]:>13.1f} {ratio:>13.1f}x')

# NB95参考値
print('\n--- NB95参考値 (CPU PyTorch, 1000件) ---')
print('  Gemma-300M: JA ~3 docs/sec, EN ~8 docs/sec')
print('  Qwen3-0.6B: JA ~1 doc/sec,  EN ~3 docs/sec')

Loading nomic-ai/nomic-embed-text-v2-moe on CPU...



--- JA ---


  3.8 docs/sec, 262.8 ms/doc
--- EN ---


  10.7 docs/sec, 93.6 ms/doc



GPU vs CPU 速度比較
         GPU (docs/s)   CPU (docs/s)  GPU/CPU ratio
--------------------------------------------------
JA               217           3.8          57.1x
EN               517          10.7          48.4x

--- NB95参考値 (CPU PyTorch, 1000件) ---
  Gemma-300M: JA ~3 docs/sec, EN ~8 docs/sec
  Qwen3-0.6B: JA ~1 doc/sec,  EN ~3 docs/sec


## 8. ITQ-LSH適合性 (ビット長最適化)

In [9]:
bit_sizes = [64, 128, 256, 512]
itq_results = []

print('ITQ-LSH ビット長最適化')
print('='*70)

# 128bit結果を保持（保存用）
best_itq = {}
best_hashes = {}

for lang, emb in [('ja', emb_ja), ('en', emb_en)]:
    print(f'\n--- {lang.upper()} (dim={emb.shape[1]}) ---')
    for n_bits in bit_sizes:
        r = quick_itq_eval(emb, MODEL_KEY, n_bits=n_bits)
        r['lang'] = lang
        itq_results.append(r)
        print(f'  ITQ-{n_bits:3d}bit: Spearman={r["spearman"]:.4f}, '
              f'Ham mean={r["ham_mean"]:.1f}, std={r["ham_std"]:.1f}')
        if n_bits == 128:
            best_itq[lang] = r['itq']
            best_hashes[lang] = r['hashes']

# NB91参考値との比較 (128bit)
print('\n--- NB91参考値 (ITQ-128bit Spearman) ---')
ref_itq = {
    'e5_base':    (-0.4715, -0.4735),
    'gemma_300m': (-0.4190, -0.4508),
    'qwen3_06b':  (-0.3919, -0.4176),
    'bge_m3':     (-0.4060, -0.4339),
    'ruri_v3':    (-0.4289, -0.3508),
}
print(f'{"model":<15} {"JA":>10} {"EN":>10}')
for m, (ja, en) in ref_itq.items():
    print(f'{m:<15} {ja:>10.4f} {en:>10.4f}')
nomic_128 = {r['lang']: r['spearman'] for r in itq_results if r['n_bits'] == 128}
print(f'{MODEL_KEY:<15} {nomic_128.get("ja", 0):>10.4f} {nomic_128.get("en", 0):>10.4f}')

ITQ-LSH ビット長最適化

--- JA (dim=768) ---
ITQ学習開始: samples=9990, dim=768, bits=64
  Centering完了: mean_norm=0.5235
  PCA完了: explained_variance=52.10%
  ITQ iteration 10: quantization_error=0.8767
  ITQ iteration 20: quantization_error=0.8758


  ITQ iteration 30: quantization_error=0.8755
  ITQ iteration 40: quantization_error=0.8754
  ITQ iteration 50: quantization_error=0.8753
ITQ学習完了
  ITQ- 64bit: Spearman=-0.5889, Ham mean=32.0, std=6.1
ITQ学習開始: samples=9990, dim=768, bits=128
  Centering完了: mean_norm=0.5235
  PCA完了: explained_variance=66.10%


  ITQ iteration 10: quantization_error=0.9016
  ITQ iteration 20: quantization_error=0.9009


  ITQ iteration 30: quantization_error=0.9006
  ITQ iteration 40: quantization_error=0.9005


  ITQ iteration 50: quantization_error=0.9004
ITQ学習完了
  ITQ-128bit: Spearman=-0.6266, Ham mean=64.0, std=9.7
ITQ学習開始: samples=9990, dim=768, bits=256
  Centering完了: mean_norm=0.5235


...

  ITQ iteration 10: quantization_error=0.9353


  ITQ iteration 20: quantization_error=0.9348


  ITQ iteration 30: quantization_error=0.9347


  ITQ iteration 40: quantization_error=0.9346


  ITQ iteration 50: quantization_error=0.9345
ITQ学習完了
  ITQ-512bit: Spearman=-0.7044, Ham mean=255.9, std=16.3

--- NB91参考値 (ITQ-128bit Spearman) ---
model                   JA         EN
e5_base            -0.4715    -0.4735
gemma_300m         -0.4190    -0.4508
qwen3_06b          -0.3919    -0.4176
bge_m3             -0.4060    -0.4339
ruri_v3            -0.4289    -0.3508
nomic_v2_moe       -0.6266    -0.6055


## 9. Embedding・ハッシュ保存

In [10]:
# Embedding保存
np.save(DATA_DIR / f'10k_{MODEL_KEY}_ja_embeddings.npy', emb_ja)
np.save(DATA_DIR / f'10k_{MODEL_KEY}_en_embeddings.npy', emb_en)
print(f'Saved: 10k_{MODEL_KEY}_ja_embeddings.npy {emb_ja.shape}')
print(f'Saved: 10k_{MODEL_KEY}_en_embeddings.npy {emb_en.shape}')

# ITQ-128bitハッシュ保存
np.save(DATA_DIR / f'10k_{MODEL_KEY}_ja_hashes_128bits.npy', best_hashes['ja'])
np.save(DATA_DIR / f'10k_{MODEL_KEY}_en_hashes_128bits.npy', best_hashes['en'])
print(f'Saved: 10k_{MODEL_KEY}_ja_hashes_128bits.npy {best_hashes["ja"].shape}')
print(f'Saved: 10k_{MODEL_KEY}_en_hashes_128bits.npy {best_hashes["en"].shape}')

# ITQモデル保存 (JA+EN混合で再学習)
emb_mixed = np.vstack([emb_ja, emb_en])
itq_mixed = ITQLSH(n_bits=128, n_iterations=50, seed=42)
itq_mixed.fit(emb_mixed)
itq_path = str(DATA_DIR / f'itq_{MODEL_KEY}_128bits.pkl')
itq_mixed.save(itq_path)
print(f'Saved: itq_{MODEL_KEY}_128bits.pkl (trained on {len(emb_mixed)} mixed samples)')

# 検証
for lang, expected in [('ja', emb_ja), ('en', emb_en)]:
    loaded = np.load(DATA_DIR / f'10k_{MODEL_KEY}_{lang}_embeddings.npy')
    assert np.allclose(loaded, expected), f'{lang} embedding mismatch!'
print('\nVerification passed.')

Saved: 10k_nomic_v2_moe_ja_embeddings.npy (9990, 768)
Saved: 10k_nomic_v2_moe_en_embeddings.npy (10000, 768)
Saved: 10k_nomic_v2_moe_ja_hashes_128bits.npy (9990, 128)
Saved: 10k_nomic_v2_moe_en_hashes_128bits.npy (10000, 128)
ITQ学習開始: samples=19990, dim=768, bits=128
  Centering完了: mean_norm=0.4424
  PCA完了: explained_variance=61.51%


  ITQ iteration 10: quantization_error=0.9015
  ITQ iteration 20: quantization_error=0.9009


  ITQ iteration 30: quantization_error=0.9006


  ITQ iteration 40: quantization_error=0.9005
  ITQ iteration 50: quantization_error=0.9004
ITQ学習完了
Saved: itq_nomic_v2_moe_128bits.pkl (trained on 19990 mixed samples)



Verification passed.


## 10. 総合サマリー

In [ ]:
print('='*90)
print(f'実験96: {MODEL_ID} 総合評価サマリー')
print('='*90)

print(f'\n--- 基本情報 ---')
print(f'  モデル:     {MODEL_ID}')
print(f'  次元:       {emb_ja.shape[1]}')
print(f'  プレフィックス: search_document: / search_query:')

print(f'\n--- Embedding品質 (cos_mean: 低いほど等方的) ---')
print(f'  JA: {result_ja["cos_mean"]:.4f}  EN: {result_en["cos_mean"]:.4f}')

print(f'\n--- STS Spearman ---')
for r in sts_results:
    print(f'  {r["dataset"]}: {r["spearman"]:.4f}')
avg_sts = np.mean([r['spearman'] for r in sts_results])
print(f'  AVG: {avg_sts:.4f}')

print(f'\n--- GPU推論速度 ---')
print(f'  JA: {speed_gpu["ja_docs_sec"]:.0f} docs/sec')
print(f'  EN: {speed_gpu["en_docs_sec"]:.0f} docs/sec')

print(f'\n--- CPU推論速度 (1000件) ---')
print(f'  JA: {res_cpu_ja["docs_per_sec"]:.1f} docs/sec')
print(f'  EN: {res_cpu_en["docs_per_sec"]:.1f} docs/sec')

print(f'\n--- ITQ-LSH品質 (Spearman) ---')
for r in itq_results:
    if r['lang'] == 'ja':
        en_r = [x for x in itq_results if x['lang'] == 'en' and x['n_bits'] == r['n_bits']][0]
        print(f'  {r["n_bits"]:3d}bit: JA={r["spearman"]:.4f}  EN={en_r["spearman"]:.4f}')

print(f'\n--- 保存ファイル ---')
for f in sorted(DATA_DIR.glob(f'*{MODEL_KEY}*')):
    print(f'  {f.name} ({f.stat().st_size / 1024:.0f} KB)')

print('\n' + '='*90)

## 評価と考察

### 1. Embedding空間の等方性: 全モデル中最良

| モデル | cos_mean JA | cos_mean EN |
|--------|-------------|-------------|
| **nomic_v2_moe** | **0.273** | **0.172** |
| gemma_300m | 0.360 | 0.206 |
| qwen3_06b | 0.494 | 0.384 |
| e5_base | 0.706 | 0.594 |
| bge_m3 | 0.605 | 0.459 |

nomic-embed-text-v2-moeはcos_meanが最も低く、**等方的なEmbedding空間**を持つ。これはLSHベースの手法（ITQ-LSH, Voronoi分割）にとって理想的な性質である。MoEアーキテクチャがベクトル空間の多様性に寄与していると考えられる。

### 2. ITQ-LSHとの相性: 圧倒的に高いハッシュ品質

| モデル | 128bit JA | 128bit EN |
|--------|-----------|-----------|
| **nomic_v2_moe** | **-0.627** | **-0.606** |
| e5_base | -0.472 | -0.474 |
| gemma_300m | -0.419 | -0.451 |
| qwen3_06b | -0.392 | -0.418 |
| bge_m3 | -0.406 | -0.434 |

|Spearman|が0.6を超えるのはnomicのみ。E5-baseに対して+33%の改善。ビット長を増やした場合も512bitで-0.70に達し、Hamming距離がコサイン類似度を高精度に近似できることを示す。

**この結果はANNベンチマーク(NB72-75)のSIFT(-0.93)に次ぐ水準であり、テキストEmbeddingとしては異例の高さ。** ITQ-LSH + Pivot方式のRecall@10向上が大きく期待できる。

### 3. STS性能: JSICKでトップ、他は中位

| Dataset | nomic | e5_base | qwen3_06b | bge_m3 |
|---------|-------|---------|-----------|--------|
| JSTS | 0.773 | 0.833 | **0.845** | 0.834 |
| JSICK | **0.817** | 0.768 | 0.793 | 0.770 |
| STS-B | 0.834 | 0.849 | **0.871** | 0.843 |
| AVG | 0.808 | 0.816 | **0.836** | 0.815 |

- JSICKでは全モデル中トップ（構文理解に強い）
- JSTS/STS-BではQwen3-0.6Bが最良、nomicはE5-baseとほぼ同水準
- STS平均0.808は十分実用的な水準

### 4. 推論速度: GPU中位、CPU非実用的

| | GPU (docs/sec) | CPU (docs/sec) |
|--|----------------|----------------|
| JA | 217 | 3.8 |
| EN | 517 | 10.7 |

- GPU速度はE5-base(413/1077)の約半分。MoEのルーティングオーバーヘッドによるもの
- CPU推論はGemma-300M(3/8)と同程度で、バッチ処理には非実用的
- **GPU前提の運用が必須**

### 5. 総合評価

| 観点 | 評価 | 備考 |
|------|------|------|
| Embedding等方性 | ★★★ | 全モデル最良 |
| ITQ-LSH相性 | ★★★ | |Spearman|>0.6、テキスト最高 |
| STS品質 | ★★☆ | AVG 0.808、JSICKトップ |
| GPU速度 | ★★☆ | E5-baseの約半分 |
| CPU速度 | ★☆☆ | 非実用的 |

**結論**: nomicはITQ-LSHおよびVoronoi分割ベースの近似最近傍検索に**最適なモデル**。等方的な空間により、ハッシュベースの候補絞り込みが高精度に機能する。STS品質は中上位で十分実用的。GPU推論が前提条件となるが、その制約下では本PoCの用途に最も適したモデルと言える。

次のステップとしてNB118でVoronoi分割の性能を検証し、等方的な空間がVoronoiのRecallにどう影響するかを確認する。